# Target API Quickstart

A short walkthrough of the new model-bound target workflow in `myflopy`.

This notebook focuses on:
- flexible `HeadTargets` inputs
- flexible `LakeStageTargets` inputs
- `model.targets...` attachment
- plotting and comparison helpers
- generating FloPy MF6 observation objects

## Imports

Start with the public package surface.

In [ ]:
import pandas as pd
import myflopy as mf

## 1. Attach head targets from GIS-style inputs

This is still the most complete pattern when you already have a points layer and a target table.

In [ ]:
head_targets = mf.HeadTargets(
    locations=head_points_gpkg,
    values=head_targets_csv,
    name_column="name",
    layer_column="layer",
    time_column="per",
)

model.targets.heads = head_targets

model.targets.heads.summary()

## 2. Attach head targets directly from model cells

Use this when you already know the target cells and do not want to build a GIS layer first.

In [ ]:
model.targets.heads = mf.HeadTargets.from_cells(
    cells=[101, 205],
    names=["MW-1", "MW-2"],
    times=[0, 1, 2],
    values={
        "MW-1": [755.2, 754.9, 754.7],
        "MW-2": [748.0, 747.8, 747.6],
    },
    time_column="per",
)

model.targets.heads.to_long().head()

## 3. Plain constructor shortcuts for simple inputs

The plain constructor now accepts lightweight mappings too, which is often convenient in scripts.

In [ ]:
model.targets.heads = mf.HeadTargets(
    locations={"MW-1": 101, "MW-2": 205},
    values={
        "per": [0, 1],
        "MW-1": [755.2, 754.9],
        "MW-2": [748.0, 747.8],
    },
    time_column="per",
)

model.targets.heads.stats()

## 4. Lake-stage targets

Lake-stage targets support named lakes with ids plus values from a file, a series, or a simple list.

In [ ]:
model.targets.lake_stage = mf.LakeStageTargets(
    locations={"deep_lake": 0},
    values=[766.1, 766.3, 766.0],
    times=[0, 1, 2],
    time_column="per",
)

model.targets.lake_stage.get()

In [ ]:
stage_series = pd.Series([766.1, 766.3, 766.0], index=[0, 1, 2], name="stage")

model.targets.lake_stage = mf.LakeStageTargets.from_series(
    lake="deep_lake",
    lake_id=0,
    values=stage_series,
    time_column="per",
)

model.targets.lake_stage.summary()

## 5. Compare, plot, and review

Once targets are attached to the model, the common review calls become model-bound.

In [ ]:
comparison = model.targets.heads.compare()
residuals = model.targets.heads.residuals()
stats = model.targets.heads.stats()

comparison.head()

In [ ]:
model.targets.heads.plot.locations()
model.targets.heads.plot.calibration()
model.targets.heads.plot.obs_vs_sim()
model.targets.heads.plot.timeseries("MW-1")

## 6. Generate FloPy MF6 observation packages

The same target definitions can now produce the lower-level FloPy observation objects.

In [ ]:
head_obs_dict = model.targets.heads.to_flopy_obs(csv_name="cumb_obs.csv")
head_obs_pkg = model.targets.heads.attach_flopy_obs(
    pname="gwf_obs",
    filename="cumb_heads.obs",
    csv_name="cumb_obs.csv",
)

lake_obs_dict = model.targets.lake_stage.to_flopy_obs(csv_name="lak_obs.csv")
lake_obs_pkg = model.targets.lake_stage.attach_flopy_obs(
    pname="lak_obs",
    filename="cumb_lak.obs",
    csv_name="lak_obs.csv",
)

head_obs_dict, lake_obs_dict

## 7. Connect the same targets to PEST

`HeadTargets` stays PEST-independent, but the PEST layer can consume it directly.

In [ ]:
pest = mf.PestProject(
    model=model,
    name="my_calibration",
    workspace=model.workspace / "pest",
    start_datetime="2024-01-01",
)

pest.add_observation(mf.HeadTargetObservationSpec(targets=model.targets.heads.targets))

## 8. Reopen a finished PEST run

Saved target metadata lets the same targets come back automatically when you reopen a completed run.

In [ ]:
run = mf.open_pest_run(completed_run_dir)
saved_targets = run.load_head_targets()
baseline_model = run.load_baseline_model()
completed_model = run.load_calibrated_model()
review = run.review()

saved_targets.summary()